# DefectVision AI - Phase 2: Anomaly Detection
## Train PatchCore on Normal/Good Images Only

**Phase 2 is fundamentally different from Phase 1:**
- Phase 1 (YOLO) needs labeled defect images with bounding boxes
- Phase 2 (PatchCore) only needs **normal/good images** -- no labels!
- It learns what "normal" looks like, then flags anything different
- Outputs a **heatmap** showing exactly where the anomaly is

This is how real factories work -- defects are rare and you can't label every possible type.

We use **Anomalib** (by Intel) which implements PatchCore, PaDiM, and many other methods.

**Run this notebook in Google Colab with GPU runtime.**

## Step 1: Install Dependencies

In [ ]:
!pip install anomalib -q

## Step 2: Check GPU

In [ ]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## Step 3: Download MVTec AD Dataset

MVTec Anomaly Detection (MVTec AD) is the gold-standard benchmark for anomaly detection.
It has 15 categories of objects/textures, each with:
- A training set of only **good/normal** images
- A test set with both good and defective images
- Pixel-level ground truth masks for defects

We'll use the **metal_nut** category (relevant to our metal domain).
Anomalib handles the download automatically.

In [ ]:
from anomalib.data import MVTec

# Choose category -- metal_nut is great for showing metal surface anomalies
# Other good options: 'screw', 'transistor', 'grid', 'tile'
CATEGORY = "metal_nut"

datamodule = MVTec(
    root="./datasets/MVTec",
    category=CATEGORY,
    image_size=(256, 256),
    train_batch_size=32,
    eval_batch_size=32,
    num_workers=4,
)

datamodule.setup()

print(f"Category: {CATEGORY}")
print(f"Training samples (good only): {len(datamodule.train_data)}")
print(f"Test samples (good + defective): {len(datamodule.test_data)}")

## Step 4: Visualize Training Data (All Normal)

Notice: the training set contains ONLY normal/good images. No defects.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Show some training images (all normal)
train_loader = datamodule.train_dataloader()
batch = next(iter(train_loader))

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for i, ax in enumerate(axes.flatten()):
    if i < len(batch["image"]):
        img = batch["image"][i].permute(1, 2, 0).numpy()
        img = np.clip(img, 0, 1)
        ax.imshow(img)
        ax.set_title("NORMAL (training)", fontsize=9, color="green")
    ax.axis('off')

plt.suptitle(f'Training Data - {CATEGORY} (all normal/good)', fontsize=14)
plt.tight_layout()
plt.show()

## Step 5: Train PatchCore Model

**PatchCore** works by:
1. Extracting feature patches from normal images using a pretrained CNN backbone
2. Building a memory bank of these "normal" features
3. At test time, comparing new image patches against the memory bank
4. High distance = anomaly

Training is fast (~5-10 minutes) because there's no gradient descent -- it just builds the memory bank.

In [ ]:
from anomalib.models import Patchcore
from anomalib.engine import Engine

# Create PatchCore model
model = Patchcore(
    backbone="wide_resnet50_2",  # Feature extraction backbone
    layers_to_extract=["layer2", "layer3"],  # Which CNN layers to use
    num_neighbors=9,
)

# Create engine and train
engine = Engine(
    max_epochs=1,  # PatchCore only needs 1 epoch (memory bank construction)
    default_root_dir="./anomaly_results",
)

print("Training PatchCore (building memory bank from normal images)...")
engine.fit(model=model, datamodule=datamodule)
print("Training complete!")

## Step 6: Evaluate and Visualize Results

Test on images that include both normal and defective samples.
The model generates:
- **Anomaly score** (0-1, higher = more anomalous)
- **Anomaly heatmap** showing where defects are

In [ ]:
# Evaluate on test set
test_results = engine.test(model=model, datamodule=datamodule)
print(f"\n=== Anomaly Detection Results ({CATEGORY}) ===")
for key, value in test_results[0].items():
    print(f"{key}: {value:.4f}")

In [ ]:
# Visualize predictions with heatmaps
from anomalib.utils.visualization import ImageVisualizer

# Get predictions on test set
predictions = engine.predict(model=model, datamodule=datamodule)

# Show some results
fig, axes = plt.subplots(3, 4, figsize=(20, 15))
fig.suptitle(f'Anomaly Detection Results - {CATEGORY}', fontsize=16)

for i in range(min(3, len(predictions))):
    pred = predictions[i]

    # Original image
    img = pred["image"][0].permute(1, 2, 0).numpy()
    img = np.clip(img, 0, 1)
    axes[i][0].imshow(img)
    axes[i][0].set_title("Original", fontsize=10)

    # Anomaly heatmap
    if "anomaly_map" in pred:
        amap = pred["anomaly_map"][0].squeeze().numpy()
        axes[i][1].imshow(amap, cmap="jet")
        axes[i][1].set_title("Anomaly Heatmap", fontsize=10)

    # Ground truth mask (if available)
    if "mask" in pred and pred["mask"] is not None:
        mask = pred["mask"][0].squeeze().numpy()
        axes[i][2].imshow(mask, cmap="gray")
        axes[i][2].set_title("Ground Truth", fontsize=10)

    # Score
    score = pred.get("pred_score", [0])[0]
    label = "ANOMALY" if pred.get("pred_label", [0])[0] else "NORMAL"
    color = "red" if label == "ANOMALY" else "green"
    axes[i][3].text(0.5, 0.5, f"{label}\nScore: {score:.3f}",
                    ha='center', va='center', fontsize=16, color=color,
                    transform=axes[i][3].transAxes)
    axes[i][3].set_title("Verdict", fontsize=10)

    for ax in axes[i]:
        ax.axis('off')

plt.tight_layout()
plt.show()

## Step 7: Export Model for Local Use

Export the trained model so we can use it in our Gradio app.

In [ ]:
import shutil
from pathlib import Path

# Export the model
engine.export(
    model=model,
    export_type="torch",
)

# Find and package the exported model
results_dir = Path("./anomaly_results")
print("\nLooking for exported model files...")

# Walk through results to find the weights
for path in results_dir.rglob("*.pt"):
    print(f"  Found: {path} ({path.stat().st_size / 1e6:.1f} MB)")

for path in results_dir.rglob("*.ckpt"):
    print(f"  Found: {path} ({path.stat().st_size / 1e6:.1f} MB)")

In [ ]:
# Package model files for download
import zipfile

# Zip the entire anomaly results directory
zip_name = "anomaly_model.zip"
with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for file_path in results_dir.rglob("*"):
        if file_path.is_file() and file_path.suffix in [".pt", ".ckpt", ".yaml", ".json"]:
            zipf.write(file_path, file_path.relative_to(results_dir))
            print(f"  Added: {file_path.relative_to(results_dir)}")

print(f"\nZipped model: {zip_name}")

# Download
try:
    from google.colab import files
    files.download(zip_name)
    print("\nDownload started! Unzip into defect-vision/models/anomaly_model/")
except ImportError:
    print("Not running in Colab - copy anomaly_model.zip manually")

## What Just Happened?

**Key takeaway for the hackathon presentation:**

1. We trained on ONLY normal/good images -- no defect labels needed
2. The model learned a "memory bank" of what normal metal nuts look like
3. At test time, it compares new images against this memory bank
4. Any region that doesn't match gets flagged with a heatmap
5. This works for **any** defect type -- even ones never seen before

This is why anomaly detection is more practical for manufacturing:
- Defects are rare (maybe 1 in 1000 items)
- You can't label every possible defect type
- New defect types appear over time
- You always have plenty of normal/good samples